# Metrics Generation for Evaluation Questions
data: 'Evaluation Questions.xlsx'
output: 'evaluation_results.csv'

In [12]:
# Imports

import json
import numpy as np
import pandas as pd

In [2]:
# Load data

DIR_PATH = '../datasets/evaluation/'

df = pd.read_excel(DIR_PATH + 'Evaluation Questions.xlsx')
df.head()

,questions,tool,arguments,model_tool,model_arguments
0,What is the busiest route in New York transit?,NaN,{},NaN,{}
1,Can you add a column to the trips table labeli...,NaN,{},NaN,{}
2,What are the five busiest routes from January ...,top_routes_by_ridership,"{'top_n': 5, 'day_code': 'WK', 'direction': 'I...",top_routes_by_ridership,"{'start_date': '2025-01-01', 'end_date': '2025..."
3,What are the busiest routes?,top_routes_by_ridership,"{'start_date': '2025-01-01', 'end_date': '2025...",top_routes_by_ridership,"{'end_date': '2025-12-31', 'top_n': 10, 'start..."
4,"Starting from March, what are the three busies...",top_routes_by_ridership,"{'start_date': '2025-03-01', 'end_date': '2025...",top_routes_by_ridership,"{'start_date': '2025-03-01', 'end_date': '2025..."


In [3]:
# Prepare data

evaluation_columns = ['questions', 'tool', 'arguments', 'model_tool', 'model_arguments']
eval = df[evaluation_columns]
eval = eval.replace(pd.NA, "None")

eval.info()
eval.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   questions        41 non-null     object
 1   tool             41 non-null     object
 2   arguments        41 non-null     object
 3   model_tool       41 non-null     object
 4   model_arguments  41 non-null     object
dtypes: object(5)
memory usage: 1.7+ KB


,questions,tool,arguments,model_tool,model_arguments
0,What is the busiest route in New York transit?,None,{},None,{}
1,Can you add a column to the trips table labeli...,None,{},None,{}
2,What are the five busiest routes from January ...,top_routes_by_ridership,"{'top_n': 5, 'day_code': 'WK', 'direction': 'I...",top_routes_by_ridership,"{'start_date': '2025-01-01', 'end_date': '2025..."
3,What are the busiest routes?,top_routes_by_ridership,"{'start_date': '2025-01-01', 'end_date': '2025...",top_routes_by_ridership,"{'end_date': '2025-12-31', 'top_n': 10, 'start..."
4,"Starting from March, what are the three busies...",top_routes_by_ridership,"{'start_date': '2025-03-01', 'end_date': '2025...",top_routes_by_ridership,"{'start_date': '2025-03-01', 'end_date': '2025..."


In [4]:
# Function to convert string representation of a dictionary to an actual dictionary

def str_to_dict(df, i : int, col_name : str) -> dict:
    d = df.loc[i, col_name].replace("'", '"')
    try:
        d = json.loads(d)
    except json.JSONDecodeError:
        print(f"Error decoding JSON for row {i}, column '{col_name}': {df.loc[i, col_name]}")
        return None
    return d

In [5]:
# Function to check if the expected arguments match the model's arguments

def match_arguments(df, i : int) -> bool:
    expected_args = str_to_dict(df, i, 'arguments')
    model_args = str_to_dict(df, i, 'model_arguments')
    
    if expected_args is None or model_args is None:
        return None
    
    for key in expected_args:
        if key not in model_args or expected_args[key] != model_args[key]:
            return False
    return True

def average_arguments(df, idx : int) -> float:
    expected_args = str_to_dict(df, idx, 'arguments')
    model_args = str_to_dict(df, idx, 'model_arguments')
    
    if expected_args is None or model_args is None:
        return None
    
    total_args = len(expected_args)
    if total_args == 0:
        return 1.0
    
    correct_count = sum(1 for key in expected_args if key in model_args and expected_args[key] == model_args[key])
    
    return correct_count / total_args

In [6]:
# Generate metrics
eval['correct_tool'] = eval['tool'] == eval['model_tool']

correct_arguments = []
for i in range(len(eval)):
    correct_arguments.append(match_arguments(eval, i))
eval['correct_arguments'] = correct_arguments

percent_arguments = []
for i in range(len(eval)):
    percent_arguments.append(average_arguments(eval, i))
eval['percent_arguments'] = percent_arguments

In [7]:
print(eval['correct_tool'].value_counts())

correct_tool
True    41
Name: count, dtype: int64


In [8]:
print(eval['correct_arguments'].value_counts())

eval[eval['correct_arguments'] == False]

correct_arguments
True     40
False     1
Name: count, dtype: int64


,questions,tool,arguments,model_tool,model_arguments,correct_tool,correct_arguments,percent_arguments
18,"From June, what are the 8 busiest stops?",busiest_stops,"{'start_date': '2025-06-01', 'end_date': '2025...",busiest_stops,"{'start_date': '2025-06-01', 'end_date': '2025...",True,False,0.666667


In [9]:
eval['percent_arguments'].describe()

count    41.000000
mean      0.991870
std       0.052058
min       0.666667
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000
Name: percent_arguments, dtype: float64

In [10]:
# Total Metrics
print(f"Total questions: {len(eval)}")
print(f"Questions yielding correct tools: {eval['correct_tool'].sum()} ({eval['correct_tool'].mean() * 100:.2f}%)")
print(f"Questions yielding all correct arguments: {eval['correct_arguments'].sum()} ({eval['correct_arguments'].mean() * 100:.2f}%)")
print(f"Average accuracy of individual arguments: {eval['percent_arguments'].mean() * 100:.2f}%")

Total questions: 41
Questions yielding correct tools: 41 (100.00%)
Questions yielding all correct arguments: 40 (97.56%)
Average accuracy of individual arguments: 99.19%


In [11]:
# Save results
eval.to_csv(DIR_PATH + 'evaluation_results.csv', index=False)
eval.to_excel(DIR_PATH + 'evaluation_results.xlsx', index=False)